# Notebook 1 — Import & Validation

Reads raw Gamry `.DTA` files from one polarization experiment, classifies each file, reconstructs the CP–EIS–CV workflow, validates consistency, and exports a processed experiment package for Notebook 2.

## What you edit
Only the configuration cell in Section 1.


In [ ]:
from pathlib import Path
import json, re, math, warnings
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)


## 1. Configuration


In [ ]:
# Change these values for each experiment.
EXPERIMENT_NAME = "Stability_Pol-curve_Ni-mesh_7M_KOH"

RAW_DATA_DIR = Path(
    r"C:\Users\edr2299\OneDrive - The University of Texas at Austin\Documents\Research\Electrochemistry Analysis\Data"
) / EXPERIMENT_NAME

OUTPUT_ROOT = Path(
    r"C:\Users\edr2299\OneDrive - The University of Texas at Austin\Documents\Research\Electrochemistry Analysis\Results"
)

PROCESSED_DIR = OUTPUT_ROOT / EXPERIMENT_NAME / "processed"

EXPERIMENT_METADATA = {
    "experiment_name": EXPERIMENT_NAME,
    "working_electrode": "Ni mesh",
    "counter_electrode": "Ni mesh",
    "reference_electrode": "Hg/HgO",
    "reference_filling_solution": "7 M KOH",
    "electrolyte": "7 M KOH",
    "koh_condition": "Fe-unpurified",
    "temperature_C": 25.0,
    "flow_rate_mL_min": 100.0,
    "geometric_area_cm2": 4.0,
}

EXPECTED = {
    "initial_eis_replicates": 3,
    "final_eis_replicates": 3,
    "step_cp_duration_s": 600.0,
    "preconditioning_duration_s": 1800.0,
    "activation_current_A": 0.060,
    "activation_duration_s": 8 * 3600,
    "eis_f_max_Hz": 100000.0,
    "eis_f_min_Hz": 0.8,
    "eis_points_per_decade": 10,
    "cv_cycles": 2,
}

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("Exists:", RAW_DATA_DIR.exists())
print("PROCESSED_DIR:", PROCESSED_DIR)


## 2. Core data structures


In [ ]:
@dataclass
class ParsedGamryFile:
    source_file: str
    role: str
    technique: str
    header: Dict[str, Any]
    tables: Dict[str, pd.DataFrame]
    sequence_index: Optional[int] = None
    current_A: Optional[float] = None
    current_density_mA_cm2: Optional[float] = None
    selected_cv_table: Optional[str] = None
    warnings: Optional[List[str]] = None

    def to_manifest(self) -> Dict[str, Any]:
        table_paths = [
            str(Path("tables") / f"{Path(self.source_file).stem}__{name}.csv")
            for name in self.tables
        ]
        return {
            "source_file": self.source_file,
            "role": self.role,
            "technique": self.technique,
            "header": self.header,
            "tables": table_paths,
            "sequence_index": self.sequence_index,
            "current_A": self.current_A,
            "current_density_mA_cm2": self.current_density_mA_cm2,
            "selected_cv_table": self.selected_cv_table,
            "warnings": self.warnings or [],
        }


## 3. Gamry text parser


In [ ]:
def _coerce_scalar(text: str) -> Any:
    text = text.strip()
    if text == "":
        return ""
    upper = text.upper()
    if upper in {"TRUE", "FALSE"}:
        return upper == "TRUE"
    try:
        if re.fullmatch(r"[-+]?\d+", text):
            return int(text)
        return float(text)
    except ValueError:
        return text


def read_text_fallback(path: Path) -> str:
    for enc in ["utf-8-sig", "utf-8", "cp1252", "latin-1"]:
        try:
            return path.read_text(encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode {path}")


def parse_header_line(line: str) -> Optional[Tuple[str, Any]]:
    parts = line.rstrip("\r\n").split("\t")
    if len(parts) < 2:
        return None
    key = parts[0].strip()
    if not key or key.upper().startswith(("CURVE", "ZCURVE", "TABLE", "OCVCURVE")):
        return None
    value = parts[2].strip() if len(parts) >= 3 and parts[2].strip() else parts[1].strip()
    unit = parts[3].strip() if len(parts) >= 4 else ""
    coerced = _coerce_scalar(value)
    return (key, {"value": coerced, "unit": unit}) if unit else (key, coerced)


def _extract_header_value(header: Dict[str, Any], key: str, default=None):
    value = header.get(key, default)
    return value.get("value", default) if isinstance(value, dict) else value


def _table_start(line: str) -> bool:
    token = line.split("\t", 1)[0].strip()
    return bool(re.fullmatch(r"(CURVE\d+|ZCURVE\d*|TABLE\d*|OCVCURVE\d*)", token, re.I))


def _find_column_header(lines: List[str], start: int) -> Optional[int]:
    common = {"Pt", "T", "Vf", "Im", "Freq", "Zreal", "Zimag", "Vdc", "Idc"}
    for j in range(start + 1, min(start + 8, len(lines))):
        cols = [x.strip() for x in lines[j].split("\t")]
        if len(set(cols) & common) >= 2:
            return j
    return None


def parse_numeric_table(lines: List[str], start: int):
    name = lines[start].split("\t", 1)[0].strip()
    header_idx = _find_column_header(lines, start)
    if header_idx is None:
        return name, pd.DataFrame(), start + 1
    columns = [x.strip() for x in lines[header_idx].split("\t")]
    rows, i = [], header_idx + 1
    while i < len(lines):
        if _table_start(lines[i]):
            break
        raw = lines[i].rstrip("\r\n")
        if not raw.strip():
            i += 1
            continue
        parts = raw.split("\t")
        if len(parts) < 2:
            break
        parts = parts[:len(columns)] + [""] * max(0, len(columns) - len(parts))
        rows.append(parts[:len(columns)])
        i += 1
    df = pd.DataFrame(rows, columns=columns)
    for col in df.columns:
        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notna().sum() >= max(1, int(0.75 * len(df))):
            df[col] = converted
    return name, df, i


def parse_gamry_dta(path: Path):
    lines = read_text_fallback(path).splitlines()
    header, tables = {}, {}
    i = 0
    while i < len(lines):
        if _table_start(lines[i]):
            name, df, i = parse_numeric_table(lines, i)
            if not df.empty:
                tables[name] = df
            continue
        parsed = parse_header_line(lines[i])
        if parsed:
            key, value = parsed
            header[key] = value
        i += 1
    return header, tables


## 4. Classification and reconstruction helpers


In [ ]:
def normalized_name(path: Path) -> str:
    return re.sub(r"[^a-z0-9#]+", "_", path.stem.lower()).strip("_")


def extract_hash_index(name: str) -> Optional[int]:
    m = re.search(r"#\s*(\d+)", name)
    return int(m.group(1)) if m else None


def classify_file(path: Path):
    name = normalized_name(path)
    if name.startswith("ocp_pre"):
        return "ocp", None
    if name.startswith("cpact"):
        return "activation_cp", None
    if name.startswith("eis_initial"):
        return "initial_eis", extract_hash_index(name)
    if name.startswith("cv_initial") or name.startswith("cv_cycle_initial"):
        return "initial_cv", None
    if name.startswith("cp_1ma_pre"):
        return "preconditioning_cp", None
    if name.startswith("eis_final"):
        return "final_eis", extract_hash_index(name)
    if name.startswith("cv_final") or name.startswith("cv_cycle_final"):
        return "final_cv", None
    if name.startswith("pc"):
        return "polarization_cp", extract_hash_index(name)
    if name.startswith("eis_pc"):
        return "polarization_eis", extract_hash_index(name)
    if name.startswith("cv") and "_pc" in name:
        return "polarization_cv", extract_hash_index(name)
    return "unclassified", extract_hash_index(name)


def detect_technique(header, tables):
    tag = str(_extract_header_value(header, "TAG", "")).upper()
    if tag:
        return tag
    cols = {c for df in tables.values() for c in df.columns}
    if {"Freq", "Zreal", "Zimag"}.issubset(cols):
        return "EIS"
    if "Vf" in cols and "Im" in cols:
        return "CV_OR_CP"
    return "UNKNOWN"


def programmed_current_A(header, tables):
    for key in ["I", "DC", "IDC", "CURRENT", "SETCURRENT"]:
        value = _extract_header_value(header, key)
        if isinstance(value, (int, float)):
            return float(value)
    for df in tables.values():
        for col in ["Idc", "Im", "I"]:
            if col in df.columns:
                s = pd.to_numeric(df[col], errors="coerce").dropna()
                if not s.empty:
                    return float(s.median())
    return None


def detect_area_cm2(header, fallback):
    for key in ["AREA", "GeoA", "GEOMAREA"]:
        value = _extract_header_value(header, key)
        if isinstance(value, (int, float)) and value > 0:
            return float(value)
    return fallback


def complete_cv_tables(tables):
    complete = []
    for name, df in tables.items():
        if len(df) < 20:
            continue
        vcol = next((c for c in ["Vf", "V"] if c in df.columns), None)
        if vcol is None:
            continue
        v = pd.to_numeric(df[vcol], errors="coerce").dropna()
        if len(v) < 20:
            continue
        dv = np.diff(v.to_numpy())
        if np.any(dv > 0) and np.any(dv < 0):
            complete.append(name)
    return complete


def parse_and_classify(path: Path, area_fallback: float):
    header, tables = parse_gamry_dta(path)
    role, seq_idx = classify_file(path)
    current = programmed_current_A(header, tables)
    area = detect_area_cm2(header, area_fallback)
    complete = complete_cv_tables(tables)
    return ParsedGamryFile(
        source_file=path.name,
        role=role,
        technique=detect_technique(header, tables),
        header=header,
        tables=tables,
        sequence_index=seq_idx,
        current_A=current,
        current_density_mA_cm2=None if current is None else current * 1000 / area,
        selected_cv_table=complete[-1] if complete else None,
        warnings=[],
    )


## 5. Parse all raw files


In [ ]:
raw_files = sorted(
    [p for p in RAW_DATA_DIR.iterdir() if p.is_file() and p.suffix.lower() == ".dta"],
    key=lambda p: p.name.lower(),
)
assert raw_files, f"No .DTA files found in {RAW_DATA_DIR}"

parsed_files, parse_failures = [], []
for path in raw_files:
    try:
        parsed_files.append(parse_and_classify(path, EXPERIMENT_METADATA["geometric_area_cm2"]))
    except Exception as exc:
        parse_failures.append({"filename": path.name, "error": repr(exc)})

print(f"Found {len(raw_files)} raw Gamry files.")
print(f"Parsed successfully: {len(parsed_files)}")
print(f"Parse failures: {len(parse_failures)}")
if parse_failures:
    display(pd.DataFrame(parse_failures))


## 6. Build polarization-step pairs


In [ ]:
def role_files(role: str):
    return [f for f in parsed_files if f.role == role]


def sort_by_current(files):
    return sorted(files, key=lambda f: (float("inf") if f.current_A is None else abs(f.current_A), f.source_file.lower()))


cp_files = sort_by_current(role_files("polarization_cp"))
eis_files = sort_by_current(role_files("polarization_eis"))
cv_files = sort_by_current(role_files("polarization_cv"))


def closest_current_match(source, candidates, used, tolerance_A=5e-4):
    available = [c for c in candidates if c.source_file not in used]
    if not available:
        return None
    src_name = normalized_name(Path(source.source_file))
    src_idx = extract_hash_index(src_name)
    src_low_loop = "_ma_" in src_name
    src_one = "1ma" in src_name
    preferred = []
    for c in available:
        cname = normalized_name(Path(c.source_file))
        if src_one and "1ma" in cname:
            preferred.append(c)
        elif src_idx is not None and extract_hash_index(cname) == src_idx and (("_ma_" in cname) == src_low_loop):
            preferred.append(c)
    pool = preferred if preferred else available
    if source.current_A is not None:
        with_current = [c for c in pool if c.current_A is not None]
        if with_current:
            best = min(with_current, key=lambda c: abs(abs(c.current_A) - abs(source.current_A)))
            if abs(abs(best.current_A) - abs(source.current_A)) <= tolerance_A:
                return best
    return pool[0] if len(pool) == 1 else None


steps, used_eis, used_cv = [], set(), set()
for step_no, cp in enumerate(cp_files, start=1):
    eis = closest_current_match(cp, eis_files, used_eis)
    cv = closest_current_match(cp, cv_files, used_cv)
    if eis:
        used_eis.add(eis.source_file)
    if cv:
        used_cv.add(cv.source_file)
    steps.append({
        "step": step_no,
        "current_A": cp.current_A,
        "current_density_mA_cm2": cp.current_density_mA_cm2,
        "cp": cp,
        "eis": eis,
        "cv": cv,
    })

step_overview = pd.DataFrame([{
    "step": s["step"],
    "current_A": s["current_A"],
    "j_mA_cm2": s["current_density_mA_cm2"],
    "cp_file": s["cp"].source_file if s["cp"] else None,
    "eis_file": s["eis"].source_file if s["eis"] else None,
    "cv_file": s["cv"].source_file if s["cv"] else None,
} for s in steps])
display(step_overview)


## 7. Validation


In [ ]:
validation_rows = []

def add_validation(level, check, filename, message):
    validation_rows.append({"level": level, "check": check, "filename": filename, "message": message})

for failure in parse_failures:
    add_validation("ERROR", "parse failure", failure["filename"], failure["error"])

for role in ["ocp", "activation_cp", "initial_cv", "preconditioning_cp", "final_cv"]:
    count = len(role_files(role))
    if count == 0:
        add_validation("ERROR", "missing role", "", f"No file classified as {role}")
    elif count > 1:
        add_validation("WARNING", "duplicate role", "", f"{count} files classified as {role}")

for role, expected_n in [("initial_eis", EXPECTED["initial_eis_replicates"]), ("final_eis", EXPECTED["final_eis_replicates"])]:
    count = len(role_files(role))
    if count != expected_n:
        add_validation("WARNING", "replicate count", "", f"{role}: expected {expected_n}, found {count}")

for s in steps:
    cp = s["cp"]
    if s["eis"] is None:
        add_validation("ERROR", "missing EIS", cp.source_file, f"No EIS matched to step {s['step']}")
    if s["cv"] is None:
        add_validation("ERROR", "missing CV", cp.source_file, f"No CV matched to step {s['step']}")
    if s["eis"] and cp.current_A is not None and s["eis"].current_A is not None:
        if abs(abs(cp.current_A) - abs(s["eis"].current_A)) > 5e-4:
            add_validation("WARNING", "CP/EIS current mismatch", s["eis"].source_file, f"CP={cp.current_A:.6g} A, EIS={s['eis'].current_A:.6g} A")

for f in parsed_files:
    area = detect_area_cm2(f.header, EXPERIMENT_METADATA["geometric_area_cm2"])
    if not np.isclose(area, EXPERIMENT_METADATA["geometric_area_cm2"], atol=1e-6):
        add_validation("WARNING", "geometric area", f.source_file, f"Detected {area:g} cm²")

for f in role_files("initial_cv") + role_files("polarization_cv") + role_files("final_cv"):
    if not f.selected_cv_table:
        add_validation("WARNING", "complete CV cycle", f.source_file, "No complete forward-and-reverse CV table detected")

for f in role_files("ocp"):
    timeout = _extract_header_value(f.header, "TIMEOUT")
    if isinstance(timeout, (int, float)) and f.tables:
        df = next(iter(f.tables.values()))
        if "T" in df.columns and len(df):
            recorded = pd.to_numeric(df["T"], errors="coerce").max()
            if np.isfinite(recorded) and recorded < 0.9 * float(timeout):
                stability = _extract_header_value(f.header, "STABILITY")
                add_validation("WARNING", "early OCP completion", f.source_file, f"Recorded {recorded:.1f} s of a {float(timeout):.1f} s maximum; stability criterion={stability!r} likely advanced the sequence")

for f in role_files("unclassified"):
    add_validation("WARNING", "unclassified file", f.source_file, "Filename pattern was not recognized")

validation = pd.DataFrame(validation_rows, columns=["level", "check", "filename", "message"])
display(validation)


## 8. Export processed experiment


In [ ]:
tables_dir = PROCESSED_DIR / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)

for f in parsed_files:
    stem = Path(f.source_file).stem
    for table_name, df in f.tables.items():
        df.to_csv(tables_dir / f"{stem}__{table_name}.csv", index=False)

roles_manifest = {}
for f in parsed_files:
    roles_manifest.setdefault(f.role, []).append(f.to_manifest())

steps_manifest = [{
    "step": s["step"],
    "current_A": s["current_A"],
    "current_density_mA_cm2": s["current_density_mA_cm2"],
    "cp": s["cp"].to_manifest() if s["cp"] else None,
    "eis": s["eis"].to_manifest() if s["eis"] else None,
    "cv": s["cv"].to_manifest() if s["cv"] else None,
} for s in steps]

manifest = {
    "schema_version": "1.0",
    "metadata": EXPERIMENT_METADATA,
    "raw_data_dir": str(RAW_DATA_DIR),
    "processed_dir": str(PROCESSED_DIR),
    "file_count": len(raw_files),
    "parsed_file_count": len(parsed_files),
    "roles": roles_manifest,
    "polarization_steps": steps_manifest,
}

with (PROCESSED_DIR / "manifest.json").open("w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
with (PROCESSED_DIR / "metadata.json").open("w", encoding="utf-8") as f:
    json.dump(EXPERIMENT_METADATA, f, indent=2)
step_overview.to_csv(PROCESSED_DIR / "polarization_step_overview.csv", index=False)
validation.to_csv(PROCESSED_DIR / "validation_report.csv", index=False)
print(f"Export complete: {PROCESSED_DIR}")


## 9. Final summary


In [ ]:
severity_counts = validation["level"].value_counts().to_dict() if not validation.empty else {}
summary = pd.DataFrame({
    "Metric": ["Raw .DTA files", "Parsed files", "Polarization steps", "Initial EIS replicates", "Final EIS replicates", "Errors", "Warnings", "Processed directory"],
    "Value": [len(raw_files), len(parsed_files), len(steps), len(role_files("initial_eis")), len(role_files("final_eis")), severity_counts.get("ERROR", 0), severity_counts.get("WARNING", 0), str(PROCESSED_DIR)],
})
display(summary)
if severity_counts.get("ERROR", 0) == 0:
    print("Experiment reconstructed successfully and is ready for Notebook 2.")
else:
    print("Experiment exported, but validation errors should be resolved before Notebook 2.")
